# Qwen3-VL Research — Colab Runner

**One-click evaluation on a Colab GPU.** Code lives on GitHub, model weights live in Google Drive, results are pushed back to GitHub.

### Before your first run (one-time setup)
1. **Runtime → Change runtime type → GPU (T4) → Save.**
2. **Add a GitHub token so Colab can push results back:**
   - On GitHub: Settings → Developer settings → Personal access tokens → *Fine-grained tokens* → generate one with **Contents: Read and write** access to the `Qwen3-VL` repo.
   - In Colab: click the **🔑 key icon** in the left sidebar → **Add new secret** → Name it `GITHUB_TOKEN`, paste the token, and toggle **Notebook access** on.

### Every session
**Runtime → Run all.** That's it. The cells, top-to-bottom:
1. Read config + GitHub token
2. Mount Google Drive (model persists here)
3. Clone or pull the latest repo from GitHub
4. Install Python dependencies
5. Download model weights to Drive (first session only, ~5 min)
6. Verify GPU + `local_transformers` wiring
7. **Run `evaluate.py`**
8. Push results back to GitHub (so they appear in your local repo on `git pull`)

---
### How data moves
```
local repo --git push--> GitHub --git pull--> Colab   (your code edits reach the GPU)
Colab runs evaluate.py -> results/<timestamp>/
Colab --git push--> GitHub --git pull--> local repo   (results come back to you)
```
- **Code** (your edits to `local_transformers/`, experiments): travels both ways through GitHub.
- **Results**: Colab commits + pushes them; you run `git pull` locally to see them.
- **Model weights** (4.5 GB): Drive only — too big for GitHub, and they persist so you only download once.

In [ ]:
# Cell 0 — Config (the only cell you might edit)

# --- Repo ---
GITHUB_USER = "adikothuri3"
REPO_NAME   = "Qwen3-VL"
GIT_BRANCH  = "main"

# --- Model ---
MODEL_REPO_ID = "Qwen/Qwen3-VL-2B-Instruct"
MODEL_PATH    = "/content/drive/MyDrive/Qwen3-VL-models/Qwen3-VL-2B-Instruct"

# --- What to run (checked in priority order) ---
# RUN_BUDGETING=True   -> Phase 5 per-group budgeting   (src/experiments/exp_budgeting.py)
# RUN_SCORING=True     -> Phase 4 within-group scoring  (src/experiments/exp_scoring.py)
# RUN_SENSITIVITY=True -> Phase 3 group ablation        (src/experiments/exp_sensitivity.py)
# RUN_INSTRUMENT=True   -> Phase 2 measurement hooks     (src/deepstack/instrument.py)
# RUN_PROBE=True        -> Phase 1 DeepStack internals    (src/deepstack/probe.py)
# all False             -> standard evaluation/profiling  (src/evaluate.py)
RUN_BUDGETING   = True
RUN_SCORING     = False
RUN_SENSITIVITY = False
RUN_INSTRUMENT  = False
RUN_PROBE       = False

# --- Evaluation args (passed to src/evaluate.py when nothing else is selected) ---
EVAL_ARGS = {
    "--device":         "cuda",
    "--dtype":          "float16",
    "--max-new-tokens": "64",
    "--num-samples":    "1",
}
EVAL_FLAGS = ["--no-torch-profiler"]  # drop this flag to enable the full torch.profiler pass

# --- Probe args (passed to src/deepstack/probe.py when RUN_PROBE=True) ---
PROBE_ARGS = {
    "--device":         "cuda",
    "--dtype":          "float16",
    "--max-new-tokens": "8",
}

# --- Instrument args (passed to src/deepstack/instrument.py when RUN_INSTRUMENT=True) ---
# num-samples=8 cycles the full real-image calibration set (no synthetic fallback).
INSTRUMENT_ARGS = {
    "--device":      "cuda",
    "--dtype":       "float16",
    "--num-samples": "8",
}
# Capturing attention saliency forces eager attention (O(seq^2), heavier on a T4
# but fine since images are capped at <=1024px). Populates the per-group
# attention distribution (otherwise attention_dist is null).
INSTRUMENT_CAPTURE_ATTENTION = True
# Images are fetched + size-capped locally; any failed fetch falls back to a
# synthetic image automatically. Set True to skip the network entirely (offline).
INSTRUMENT_SYNTHETIC_ONLY = False

# --- Sensitivity args (passed to src/experiments/exp_sensitivity.py when RUN_SENSITIVITY=True) ---
# Phase 3 / Experiment 2: ablate each DeepStack group (8 conditions) and score
# labeled accuracy + first-token KL across task types. num-samples is PER TASK,
# PER CONDITION -> total generations = num-samples * num_tasks * 8.
# Smoke-test with a small num-samples (e.g. 3) on the first run, then scale to 100.
SENSITIVITY_ARGS = {
    "--device":         "cuda",
    "--dtype":          "float16",
    "--num-samples":    "100",
    "--max-new-tokens": "32",
    # "--tasks":        "general_vqa,textvqa,docvqa,counting",  # default = all four
}

# --- Scoring args (passed to src/experiments/exp_scoring.py when RUN_SCORING=True) ---
# Phase 4b / Experiment 4: for each task, compare within-group token scorers at
# several keep-ratios, using a UNIFORM budget across all 3 DeepStack groups (same
# keep-ratio per group) so the comparison isolates scorer quality from budget
# allocation (that allocation is Phase 5). Default methods = random (control),
# activation_magnitude, hybrid, vision_attention (the VisPruner/FasterVLM
# vision-encoder signal; capture is automatic when vision_attention is selected).
# num-samples is PER TASK; total generations = num-samples * num_tasks *
# (1 + num_methods * num_pruning_ratios). With the defaults: 300 * 4 * (1 + 4*2)
# = 10,800 generations. Results checkpoint to scoring.json after each task.
# Smoke-test with a small num-samples (e.g. 3) on the first run, then scale to 300.
SCORING_ARGS = {
    "--device":         "cuda",
    "--dtype":          "float16",
    "--num-samples":    "300",
    "--max-new-tokens": "32",
    # "--keep-ratios":  "1.0,0.50,0.25",                                          # default
    # "--methods":      "random,activation_magnitude,hybrid,vision_attention",    # default
    #   (also available: spatial_uniform, diversity — dropped after run 1 as proven losers)
    # "--tasks":        "general_vqa,textvqa,docvqa,counting",                    # default = all four
}

# --- Budgeting args (Phase 5 / Stage A; src/experiments/exp_budgeting.py when RUN_BUDGETING=True) ---
# Stage A finds, per task, the optimal per-DeepStack-group keep-budget + within-group scorer by
# ZEROING (diagnostic; no latency change — that is Stage B). Two modes; one file per task.
#
# >>> CURRENT STEP: VALIDATE. The 3-task sweep is DONE (results/20260604_232301; n=100). <<<
#   validate water-fills the measured sweep curves into joint per-group budgets (r0,r1,r2) and runs
#   them head-to-head vs UNIFORM (T,T,T) and a flat GLOBAL-TOPK baseline on a DISJOINT held-out split,
#   all at an EQUAL retained-token count, with bootstrap 95% CIs. Auto-finds the latest sweep dir.
#   --scorer pins the within-group scorer to `hybrid` (the Phase-4b feature-based winner). Do NOT let
#   it auto-pick: accuracy at n=100 is noise and the auto-picker chose random/vision_attention per task.
BUDGETING_MODE = "validate"
BUDGETING_ARGS = {
    "--device":         "cuda",
    "--dtype":          "float16",
    "--max-new-tokens": "32",
    "--num-samples":    "300",                 # held-out; auto-skips the first 100 (the sweep split)
    "--targets":        "0.5,0.3,0.2,0.15",    # aggressive — where TextVQA's per-group win should appear
    "--scorer":         "hybrid",              # PIN the feature-based scorer (not the accuracy-noise auto-pick)
    # "--sweep-dir":    "results/20260604_232301",  # else auto = latest dir with budgeting_sweep__*.json
    # "--n-candidates": "3",                    # per-group budget candidates per target
    #
    # --- To run another SWEEP instead: set BUDGETING_MODE="sweep" and swap in these args ---
    # "--num-samples":  "100",   # PER SAMPLE; ~69k gens across 3 tasks -> use an A100, a few hours
    # "--tasks":        "general_vqa,docvqa,textvqa",
    # "--scorers":      "random,activation_magnitude,hybrid,vision_attention",
    # "--groups":       "0,1,2",
    # "--step":         "0.05",
}

# --- Push results back to GitHub when the run finishes? ---
PUSH_RESULTS = True

# --- GitHub token (read from Colab Secrets — never hardcode it here) ---
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    print("GitHub token loaded from Colab Secrets.")
except ImportError:
    print("Not running in Colab — `google.colab` unavailable. "
          "Clone/pull of a public repo still works, but pushing results will be skipped.")
except userdata.SecretNotFoundError:
    print("Secret 'GITHUB_TOKEN' does not exist. Add it via the 🔑 panel (left sidebar). "
          "Pushing results will be skipped.")
except userdata.NotebookAccessError:
    print("Secret 'GITHUB_TOKEN' exists but notebook access is OFF. "
          "Open the 🔑 panel and toggle 'Notebook access' on for GITHUB_TOKEN, then re-run this cell. "
          "Pushing results will be skipped until then.")
except Exception as e:
    print(f"Unexpected error reading GITHUB_TOKEN: {type(e).__name__}: {e}. "
          "Pushing results will be skipped.")

REPO_DIR = f"/content/{REPO_NAME}"
# Authenticated URL is used only in subprocess calls, never printed.
_auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
REPO_URL_AUTH = f"https://{_auth}github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_URL_SAFE = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
if RUN_BUDGETING:
    _mode = f"BUDGETING (Phase 5 Stage A — {BUDGETING_MODE})"
elif RUN_SCORING:
    _mode = "SCORING (Phase 4b within-group scoring)"
elif RUN_SENSITIVITY:
    _mode = "SENSITIVITY (Phase 3 group ablation)"
elif RUN_INSTRUMENT:
    _mode = "INSTRUMENT (Phase 2 measurement hooks)"
elif RUN_PROBE:
    _mode = "PROBE (Phase 1 DeepStack internals)"
else:
    _mode = "EVAL (evaluate.py)"
print(f"Repo: {REPO_URL_SAFE}  (branch: {GIT_BRANCH})")
print(f"Mode: {_mode}")

In [8]:
# Cell 1 — Mount Google Drive (model weights persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Cell 2 — Clone or pull the latest repo from GitHub
import os, subprocess

def _git(args, **kw):
    """Run a git command; never echoes the token-bearing URL."""
    r = subprocess.run(["git", *args], capture_output=True, text=True, **kw)
    out = (r.stdout + r.stderr)
    if GITHUB_TOKEN:
        out = out.replace(GITHUB_TOKEN, "***")
    return r.returncode, out.strip()

if os.path.exists(f"{REPO_DIR}/.git"):
    _git(["-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL_AUTH])
    code, out = _git(["-C", REPO_DIR, "pull", "origin", GIT_BRANCH])
else:
    code, out = _git(["clone", "--branch", GIT_BRANCH, REPO_URL_AUTH, REPO_DIR])
print(out or "(no output)")

os.chdir(REPO_DIR)
print(f"\nWorking dir: {os.getcwd()}")
assert code == 0, "git clone/pull failed — check the output above."

KeyboardInterrupt: 

In [ ]:
# Cell 3 — Install dependencies
import subprocess, sys

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True,
)
out = (r.stdout + r.stderr).strip()
print(out[-2000:] if len(out) > 2000 else out or "All packages installed.")
if r.returncode != 0:
    print("\n[warning] pip returned a non-zero exit code — review the output above.")

All packages installed.


In [ ]:
# Cell 4 — Download model weights to Drive (first session only, ~4.5 GB / ~5 min)
import os

if os.path.isdir(MODEL_PATH) and os.listdir(MODEL_PATH):
    print(f"Model already in Drive — skipping download.\n{MODEL_PATH}")
else:
    print(f"Downloading {MODEL_REPO_ID} to Drive (~4.5 GB) ...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=MODEL_PATH,
        ignore_patterns=["*.pt", "*.bin"],  # safetensors only
    )
    print("\nDownload complete.")

Model already in Drive — skipping download.
/content/drive/MyDrive/Qwen3-VL-models/Qwen3-VL-2B-Instruct


In [ ]:
# Cell 5 — Verify GPU and local_transformers wiring
import torch, sys, inspect

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU            : {props.name}")
    print(f"VRAM           : {props.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Runtime -> Change runtime type -> GPU (T4) -> Save, then Run all.")

# evaluate.py wires this itself; we verify here so a misconfig fails loudly and early.
sys.path.insert(0, REPO_DIR)
import local_transformers
sys.modules["transformers"] = local_transformers
from local_transformers.models.qwen3_vl.modeling_qwen3_vl import Qwen3VLForConditionalGeneration

src = inspect.getfile(Qwen3VLForConditionalGeneration)
print(f"\nModel source   : {src}")
assert f"{REPO_DIR}/local_transformers" in src, "ERROR: not using local source!"

CUDA available : True
GPU            : Tesla T4
VRAM           : 15.6 GB

Model source   : /content/Qwen3-VL/local_transformers/models/qwen3_vl/modeling_qwen3_vl.py


In [ ]:
# Cell 6 — Run the model  (the main event)
# Dispatches in priority order: Phase 5 budgeting, Phase 4 scoring, Phase 3 sensitivity, Phase 2 instrument, Phase 1 probe, or evaluate.py.
import glob
import os
import subprocess, sys

if RUN_BUDGETING:
    # Phase 5 Stage A per-group budgeting. Runs over --tasks (default general_vqa,docvqa,textvqa),
    # one file per task: budgeting_sweep__<task>.json / budgeting_validation__<task>.json.
    # validate auto-finds the latest sweep dir unless you pass --sweep-dir in BUDGETING_ARGS.
    cmd = [sys.executable, "-m", "src.experiments.exp_budgeting", BUDGETING_MODE, "--model-id", MODEL_PATH]
    for k, v in BUDGETING_ARGS.items():
        cmd += [k, v]
elif RUN_SCORING:
    # Phase 4 within-group scoring. Writes results/<ts>/scoring.json.
    cmd = [sys.executable, "-m", "src.experiments.exp_scoring", "--model-id", MODEL_PATH]
    for k, v in SCORING_ARGS.items():
        cmd += [k, v]
elif RUN_SENSITIVITY:
    # Phase 3 ablation. Writes results/<ts>/sensitivity.json.
    cmd = [sys.executable, "-m", "src.experiments.exp_sensitivity", "--model-id", MODEL_PATH]
    for k, v in SENSITIVITY_ARGS.items():
        cmd += [k, v]
elif RUN_INSTRUMENT:
    # Run as a module (-m) so `from src.evaluate import ...` resolves with the
    # repo root on sys.path. Writes results/<ts>/deepstack_instrument.json.
    cmd = [sys.executable, "-m", "src.deepstack.instrument", "--model-id", MODEL_PATH]
    for k, v in INSTRUMENT_ARGS.items():
        cmd += [k, v]
    if INSTRUMENT_CAPTURE_ATTENTION:
        cmd += ["--capture-attention"]
    if INSTRUMENT_SYNTHETIC_ONLY:
        cmd += ["--synthetic-only"]
elif RUN_PROBE:
    # Writes results/<ts>/deepstack_probe.json.
    cmd = [sys.executable, "-m", "src.deepstack.probe", "--model-id", MODEL_PATH]
    for k, v in PROBE_ARGS.items():
        cmd += [k, v]
else:
    cmd = [sys.executable, "src/evaluate.py", "--model-id", MODEL_PATH]
    for k, v in EVAL_ARGS.items():
        cmd += [k, v]
    cmd += EVAL_FLAGS

print("Running:", " ".join(cmd), "\n")
# capture_output unset -> output streams live into the cell.
proc = subprocess.run(cmd, cwd=REPO_DIR)
print(f"\nProcess exited with code {proc.returncode}")
assert proc.returncode == 0, "run failed — review the log above."

# After a successful run, render the figures + EXPLAINER so they get committed and
# pushed alongside the JSON. (No model load — reads JSON only.)
if RUN_BUDGETING:
    # One sweep file per task; pair each with its latest validation file (if any).
    # Figures go to results/<sweep_ts>/figures/<task>/ so the tasks don't overwrite each other.
    def _task_of(p, prefix):
        return os.path.basename(p).replace(prefix, "").replace(".json", "")
    sweep_by_task = {}
    for s in sorted(glob.glob(f"{REPO_DIR}/results/*/budgeting_sweep__*.json")):
        sweep_by_task[_task_of(s, "budgeting_sweep__")] = s  # sorted asc -> latest wins
    val_by_task = {}
    for v in sorted(glob.glob(f"{REPO_DIR}/results/*/budgeting_validation__*.json")):
        val_by_task[_task_of(v, "budgeting_validation__")] = v
    for task, s in sweep_by_task.items():
        figdir = os.path.join(os.path.dirname(s), "figures", task)
        vizcmd = [sys.executable, "-m", "src.deepstack.visualize_budgeting", s, "--output-dir", figdir]
        if task in val_by_task:
            vizcmd += ["--validation", val_by_task[task]]
        print(f"\nVisualizing [{task}] {s}")
        viz = subprocess.run(vizcmd, cwd=REPO_DIR)
        print(f"  exited {viz.returncode}")

if RUN_SCORING:
    jsons = sorted(glob.glob(f"{REPO_DIR}/results/*/scoring.json"))
    if jsons:
        latest_json = jsons[-1]
        print(f"\nVisualizing {latest_json}")
        viz = subprocess.run(
            [sys.executable, "-m", "src.deepstack.visualize_scoring", latest_json], cwd=REPO_DIR
        )
        print(f"Visualization exited with code {viz.returncode}")

if RUN_SENSITIVITY:
    jsons = sorted(glob.glob(f"{REPO_DIR}/results/*/sensitivity.json"))
    if jsons:
        latest_json = jsons[-1]
        print(f"\nVisualizing {latest_json}")
        viz = subprocess.run(
            [sys.executable, "-m", "src.deepstack.visualize_sensitivity", latest_json], cwd=REPO_DIR
        )
        print(f"Visualization exited with code {viz.returncode}")

if RUN_INSTRUMENT:
    jsons = sorted(glob.glob(f"{REPO_DIR}/results/*/deepstack_instrument.json"))
    if jsons:
        latest_json = jsons[-1]
        print(f"\nVisualizing {latest_json}")
        viz = subprocess.run(
            [sys.executable, "-m", "src.deepstack.visualize", latest_json], cwd=REPO_DIR
        )
        print(f"Visualization exited with code {viz.returncode}")

In [ ]:
# Cell 7 — Push results back to GitHub
# Commits the newest results/<timestamp>/ dir and pushes it so it lands in your
# local repo on the next `git pull`. Large Chrome traces (trace_*.json) are
# gitignored, so only the JSON metrics + chart are pushed.
import os, glob

if not PUSH_RESULTS:
    print("PUSH_RESULTS is False — skipping.")
elif not GITHUB_TOKEN:
    print("No GITHUB_TOKEN — skipping push. Results remain in Colab at results/ only.")
else:
    runs = sorted(glob.glob(f"{REPO_DIR}/results/*/"))
    if not runs:
        print("No results/ runs found — nothing to push.")
    else:
        latest = runs[-1]
        _git(["-C", REPO_DIR, "config", "user.name",  "colab-runner"])
        _git(["-C", REPO_DIR, "config", "user.email", "colab@users.noreply.github.com"])
        _git(["-C", REPO_DIR, "add", "results/"])
        run_name = os.path.basename(latest.rstrip("/"))
        code, out = _git(["-C", REPO_DIR, "commit", "-m", f"Colab run: results/{run_name}"])
        print(out or "(nothing to commit)")
        if code == 0:
            code, out = _git(["-C", REPO_DIR, "push", "origin", GIT_BRANCH])
            print(out or "(push complete)")
            print("\nDone. Run `git pull` in your local repo to get these results."
                  if code == 0 else "\nPush failed — check token permissions.")
        else:
            print("Nothing new to commit (results may already be pushed).")

No GITHUB_TOKEN — skipping push. Results remain in Colab at results/ only.
